In [1]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 75.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from threading import Thread

# 1. Load the model
# This model is natively multimodal and uses a hybrid Gated DeltaNet architecture
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3.5-4B",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [3]:
# Urdu prompt
messages = [
    {"role": "user", "content": "مرغی پر ایک مزاحیہ لطیفہ بنائیں۔"}
]

# Apply chat template
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [4]:
from transformers import TextIteratorStreamer

# Create streamer
streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

# Generation parameters
generation_kwargs = dict(
    inputs,
    streamer=streamer,
    max_new_tokens=500,
    do_sample=False
)

# Start generation in a separate thread
thread = Thread(target=model.generate, kwargs=generation_kwargs)
thread.start()

# Print tokens as they're generated
for new_text in streamer:
    print(new_text, end="", flush=True)

thread.join()

Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


ایک دفعہ کا ذکر ہے کہ ایک مرغی پر ایک بہت ہی بھوکا اور پھٹپٹا ہوا تھا۔ وہ زمین پر کھڑا تھا اور آسمان کی طرف دیکھ رہا تھا۔

اچانک اس کے سامنے ایک بڑا کھانا پکانے والا آ گیا اور اس نے مرغی کو دیکھتے ہوئے کہا: "وہاں مرغی! تم کھانا کھانے کے لیے یہاں آؤ، میں تمہیں بہترین کھانا دیتا ہوں۔"

مرغی نے خوشی سے کہا: "بہت شکریہ! میں فوراً آتا ہوں۔"

جب مرغی کھانے کے قریب پہنچا تو کھانا پکانے والا اسے دیکھتے ہوئے پوچھا: "تم کھانے کے لیے آئے ہو؟"

مرغی نے جواب دیا: "جی ہاں، میں کھانے کے لیے آ رہا ہوں۔"

کھانا پکانے والا مسکرا کر کہا: "بہت اچھا! تو اب تمہیں کھانے کے لیے کھانا دیتا ہوں۔"

مرغی نے خوشی سے کہا: "شکریہ! میں کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لیے کھانے کے لی